In [1]:
# Cell 1 — MiVOLO v2 dependencies (correct version)
!pip install -q git+https://github.com/WildChlamydia/MiVOLO.git
!pip install -q transformers==4.51.0 accelerate==1.8.1 timm einops
!pip install -q numpy==1.26.4

  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2

In [2]:
# Cell 2 — Drive + imports
import os, sys, torch, cv2, numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Notebook4"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Device: {DEVICE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Device: cuda


In [3]:
# Cell 3 — Load MiVOLO v2
from transformers import AutoModelForImageClassification, AutoConfig, AutoImageProcessor

config = AutoConfig.from_pretrained(
    "iitolstykh/mivolo_v2", trust_remote_code=True)

mivolo = AutoModelForImageClassification.from_pretrained(
    "iitolstykh/mivolo_v2",
    trust_remote_code=True,
    torch_dtype=torch.float16
).to(DEVICE).eval()

processor = AutoImageProcessor.from_pretrained(
    "iitolstykh/mivolo_v2", trust_remote_code=True)

print("✓ MiVOLO v2 loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

mivolo_image_processor.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/iitolstykh/mivolo_v2:
- mivolo_image_processor.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


✓ MiVOLO v2 loaded


In [4]:
# Cell 4 — Unique child keys (train + val + test)
import pandas as pd

DRIVE = "/content/drive/MyDrive/Notebook4"

train_df = pd.read_csv(f"{DRIVE}/data/splits/train_identity.csv")
val_df   = pd.read_csv(f"{DRIVE}/data/splits/val_identity.csv")
test_df  = pd.read_csv(f"{DRIVE}/data/splits/test_identity.csv")

all_keys = set(train_df["p2"].tolist() + 
               val_df["p2"].tolist() + 
               test_df["p2"].tolist())

print(f"✓ Unique child keys: {len(all_keys):,}")

✓ Unique child keys: 1,967


In [20]:
print(outputs.age_output[0].item())

34.4375


In [21]:
# Cell 5 FINAL — full loop
from tqdm import tqdm

FIW_ROOT   = f"{DRIVE}/data/raw/FIDs"
OUTPUT_CSV = f"{DRIVE}/data/ages/mivolo_ages.csv"
os.makedirs(f"{DRIVE}/data/ages", exist_ok=True)

results, errors = [], []

for key in tqdm(all_keys, desc="MiVOLO"):
    img_path = os.path.join(FIW_ROOT, key)
    if not os.path.exists(img_path):
        errors.append((key, "no_existe"))
        continue
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        errors.append((key, "cv2_none"))
        continue
    try:
        face_input = processor.preprocess([img_bgr])["pixel_values"].to(DEVICE).half()
        body_input = torch.zeros_like(face_input)
        with torch.no_grad():
            outputs = mivolo(faces_input=face_input, body_input=body_input)
        age = round(outputs.age_output[0].item())
        results.append({"key": key, "age": age})
    except Exception as e:
        errors.append((key, str(e)))

ages_df = pd.DataFrame(results)
ages_df.to_csv(OUTPUT_CSV, index=False)
print(f"✓ Saved: {OUTPUT_CSV}")
print(f"  Procesadas: {len(results):,} | Errores: {len(errors):,}")
if errors:
    print(f"  Primeros 5 errores: {errors[:5]}")
print(ages_df["age"].describe())

MiVOLO: 100%|██████████| 1967/1967 [00:55<00:00, 35.73it/s]

✓ Saved: /content/drive/MyDrive/Notebook4/data/ages/mivolo_ages.csv
  Procesadas: 1,967 | Errores: 0
count    1967.000000
mean       18.953737
std        14.312932
min         0.000000
25%         6.000000
50%        17.000000
75%        29.000000
max        87.000000
Name: age, dtype: float64
